# 07. Data Encoding and Splitting

Encode categorical variables and split data for machine learning.

## Objectives
- Load standardized features from previous step
- Analyze and encode categorical data (multi-hot and one-hot)
- Split data into train/test/validation sets

## Table of Contents
1. [Load Data](#load-data)
2. [Point-in-Time Filtering](#point-in-time-filtering)
3. [Event Categories Analysis & Feature Encoding](#event-categories-analysis--feature-encoding)
   - 3.1 [Event Categories Discovery](#event-categories-discovery)
   - 3.2 [Multi-Hot Encoding for Events](#multi-hot-encoding-for-events)
   - 3.3 [One-Hot Encoding for Categorical Variables](#one-hot-encoding-for-categorical-variables)
4. [Data Splitting](#data-splitting)
5. [Missing Value Analysis and Imputation](#missing-value-analysis-and-imputation)

6. [Data Standardization](#data-standardization)---

7. [Save Dataset](#save-dataset)

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [ ]:
# Define paths
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
OUT_DIR = PROCESSED_DIR

# Load derived features
data = pd.read_csv(PROCESSED_DIR / "features_standardized.csv", sep=None, engine="python", encoding="utf-8-sig")
print(f"Data shape: {data.shape}")
data.head()

## 2. Point-in-Time Filtering

Apply temporal filtering to exclude students who dropped out before the prediction cutoff date.

### 2.1 December 5th Cutoff Strategy

The model performs a point-in-time prediction: at December 5th it predicts the probability that a student will have dropped out at the end of the college year. 

**Rationale for December 5th cutoff:**
- Includes student results data from Block A (typically runs until Nov 8-14)
- Exams are taken in the last week of a block
- Results are final within 3 weeks after exam (sometimes longer)
- December 5th provides sufficient margin for most exam results to be available

**Filtering approach:** Only include students who are still studying (i.e., did not drop out before Dec 5th) to avoid bias in the collected data.

In [ ]:
# Create Dec 5th cutoff for each college year
# Filter students who dropped out after Dec 5th of their college year
# (or still enrolled, as their dropout_date would be after the cutoff)
data_filtered = data[
    pd.to_datetime(data['dropout_date']) > 
    pd.to_datetime(data['sdo5_degree_COLLEGEJAAR'].astype(str) + '-12-05')
].copy()

print(f"Original dataset: {len(data)} students")
print(f"Filtered dataset: {len(data_filtered)} students (excluded students dropped out before Dec 5th)")

# Replace data with filtered dataset
data = data_filtered
del data_filtered

# Remove dropout_date
data = data.drop(columns=['dropout_date'])

## 3. Event Categories Analysis & Feature Encoding

Analyze orientation events data and apply encoding techniques for all categorical variables.

### 3.1 Event Categories Discovery

Explore and clean the orientation events data to understand what categories exist.

In [ ]:
# Clean and analyze event categories
event_series = (
    data['sdo2_orientation_Event_Types_Attended']
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace("online opleidingspresentatie", "online_opleidingspresentatie", regex=False)
    .str.replace("open dag", "open_dag", regex=False)
    .str.split(r'\s*,\s*')
    .explode()
    .str.strip()
)

print("Unique events:", sorted(event_series.unique()))
print("Event counts:\n", event_series.value_counts())

### 3.2 Multi-Hot Encoding for Events

Convert comma-separated event values into binary indicator columns.

In [ ]:
# Multi-hot encoding for event categories
expected_events = sorted(event_series.unique())
expected_cols = [(e if e == "absent" else f"attended_{e}") for e in expected_events]

if all(col in data.columns for col in expected_cols):
    print("✓ Multi-hot encoding already done. Skipping.")
else:
    # Create multi-hot matrix
    tmp = event_series.reset_index()
    tmp.columns = ["row_id", "event"]
    multi_hot = pd.crosstab(tmp["row_id"], tmp["event"]).astype("Int64")
    
    # Rename columns: "absent" stays as-is, others get "attended_" prefix
    rename_map = {c: (c if c == "absent" else f"attended_{c}") for c in multi_hot.columns}
    multi_hot.rename(columns=rename_map, inplace=True)
    
    # Join and fill missing values
    data = data.join(multi_hot, how="left")
    data[list(multi_hot.columns)] = data[list(multi_hot.columns)].fillna(0).astype("Int64")
    print("✓ Created event columns:", list(multi_hot.columns))

# Remove original column
data.drop(columns=["sdo2_orientation_Event_Types_Attended"], inplace=True, errors="ignore")

### 3.3 One-Hot Encoding for Categorical Variables

Apply standard one-hot encoding to remaining categorical columns.

In [ ]:
# One-hot encode remaining categorical columns
encode_cols = [
    'sdo5_degree_degree',
    'sdo1_previous_Previous_Education_Type', 
    'sdo2_skc_ADVIES_DEF',
    'age_category'
]

data = pd.get_dummies(data, columns=encode_cols, drop_first=False, dtype='int')
print(f"Final data shape: {data.shape}")

## 4. Data Splitting

Data will be split using a hybrid-temporal approach:
- Training: 2021 and 55% of 2022
- Validation: 45% of 2022
- Test: 2023

In [ ]:
# Assign data splits based on year
data['set'] = np.where(data['sdo5_degree_COLLEGEJAAR'] == 2021, 'train',
                np.where(data['sdo5_degree_COLLEGEJAAR'] == 2023, 'test', 'val'))
random_2022 = data[data['sdo5_degree_COLLEGEJAAR'] == 2022].sample(frac=0.55, random_state=42).index
data.loc[random_2022, 'set'] = 'train'
data.loc[(data['sdo5_degree_COLLEGEJAAR'] == 2022) & (~data.index.isin(random_2022)), 'set'] = 'val'

# Print split counts
print("Data split counts:\n", data['set'].value_counts())

Note: for now a single split is done (i.e. one random state), but in the future the random sampling of the 2022 data will be repeated for different random states. 

## 5. Missing Value Analysis and Imputation

**Purpose**: Analyze and handle missing values in the dataset before standardization to ensure proper scaling.

In [ ]:
# Comprehensive missing value analysis
print("MISSING VALUE ANALYSIS:")
print("=" * 50)

# Check for missing values in the dataset
missing_counts = data.isnull().sum().sum()
print(f"Total missing values in dataset: {missing_counts}")

if missing_counts > 0:
    print("\nColumns with missing values:")
    missing_cols = data.isnull().sum()[data.isnull().sum() > 0]
    print(missing_cols.sort_values(ascending=False))
    
    print("\nMissing value percentages:")
    missing_percentages = (data.isnull().sum() / len(data) * 100)
    missing_percentages = missing_percentages[missing_percentages > 0].sort_values(ascending=False)
    for col, pct in missing_percentages.items():
        print(f"  {col}: {pct:.2f}%")
        
else:
    print("✅ No missing values found in the dataset!")

In [ ]:
# Handle missing values with imputation
if missing_counts > 0:
    print("\nAPPLYING IMPUTATION:")
    print("=" * 30)
    print("Strategy: Mean imputation for numerical features")
    print("Note: Fitting imputer only on training set to prevent data leakage")
    
    # Identify numerical columns that need imputation
    numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    # Remove ID and target columns
    exclude_from_imputation = ['SINH_ID', 'sdo5_degree_drop_out', 'set', 'sdo5_degree_COLLEGEJAAR']
    numerical_cols = [col for col in numerical_cols if col not in exclude_from_imputation]
    
    # Find which numerical columns actually have missing values
    cols_with_missing = []
    for col in numerical_cols:
        if data[col].isnull().sum() > 0:
            cols_with_missing.append(col)
    
    print(f"\nColumns to be imputed: {cols_with_missing}")
    
    if cols_with_missing:
        # Fit imputer only on training set to prevent data leakage
        from sklearn.impute import SimpleImputer
        train_data = data[data['set'] == 'train']
        
        # Create and fit imputer
        imputer = SimpleImputer(strategy='mean')
        imputer.fit(train_data[cols_with_missing])
        
        # Apply imputation to all data
        data[cols_with_missing] = imputer.transform(data[cols_with_missing])
        
        print(f"✅ Imputation completed!")
        print(f"Remaining missing values: {data.isnull().sum().sum()}")
        
        # Save imputation statistics for reference
        print("\nImputation values used (means from training set):")
        for i, col in enumerate(cols_with_missing):
            print(f"  {col}: {imputer.statistics_[i]:.6f}")
    else:
        print("No numerical columns with missing values found.")
else:
    print("\n✅ No imputation needed - dataset is complete!")

print(f"\nFinal dataset shape: {data.shape}")
print(f"Final missing value count: {data.isnull().sum().sum()}")

## 6. Data Standardization
Scale all numerical features so they have similar ranges, which will help some ML algorithms perform better. 
Approach: fit scaler only on training set, then apply on all data sets (train, validate, test). 

### 6.1 Investigate which columns to standardize

In [ ]:
# Select training dataset
data_train = data[data['set'] == 'train']

# Find all float features
float_cols = data_train.select_dtypes(include=[np.float64]).columns.tolist()

# Remove columns that shouldn't be scaled
exclude_cols = ['SINH_ID', 'sdo5_degree_COLLEGEJAAR', 'sdo5_degree_drop_out', 'sdo2_orientation_Number_of_Event_Types', 
                'sdo2_orientation_Number_of_Events_Attended']
float_cols = [col for col in float_cols if col not in exclude_cols]

# Summary
print(f"Number of float columns: {len(float_cols)}")
print(f"Float columns to standardize: {float_cols}")

# Look visually at distributions before standardization
import matplotlib.pyplot as plt
import seaborn as sns
for col in float_cols:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(data_train[col].dropna(), kde=True)
    plt.title(f"Before Standardization: {col}")
    
    plt.subplot(1,2,2)
    sns.boxplot(x=data_train[col].dropna())
    plt.title(f"Before Standardization: {col}")
    
    plt.show()

### 6.2 Standardize float columns
Fit scalar first on training set, then apply on all data. 

In [ ]:
# Define which columns to standardize using RobustScaler
float_cols_robust = ['gap_prev_exam_to_start_weeks', 'sdo1_postal_distance_distance_to_3584_CS', 'sdo1_prev_distance_distance_to_3584_CS']
from sklearn.preprocessing import RobustScaler
robust_scaler = RobustScaler()
data_train.loc[:, float_cols_robust] = robust_scaler.fit_transform(data_train[float_cols_robust])

# Other float columns to standardize with StandardScaler
float_cols_standard = [col for col in float_cols if col not in float_cols_robust]

# Standardize float features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
data_train.loc[:, float_cols_standard] = scaler.fit_transform(data_train[float_cols_standard])

# Apply same scaling transformation to full dataset
data.loc[:, float_cols_robust] = robust_scaler.transform(data[float_cols_robust])
data.loc[:, float_cols_standard] = scaler.transform(data[float_cols_standard])

In [ ]:
# Look visually at distributions after standardization
import matplotlib.pyplot as plt
import seaborn as sns
for col in float_cols:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(data_train[col].dropna(), kde=True)
    plt.title(f"Before Standardization: {col}")
    
    plt.subplot(1,2,2)
    sns.boxplot(x=data_train[col].dropna())
    plt.title(f"Before Standardization: {col}")
    
    plt.show()

## 7. Save Dataset

In [ ]:
# Save dataset
output_path = OUT_DIR / "encoded_and_split_data.csv"
data.to_csv(output_path, index=False)

print("✅ Data encoding and splitting complete!")
print(f"   📂 Saved: {output_path}")
print(f"   📊 Shape: {data.shape}")
print(f"   🏷️ Columns: {len(data.columns)}")

# Show data type summary
print(f"\n📋 Data type summary:")
dtype_counts = data.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"   {dtype}: {count} columns")